# Surface Scattering and BSDF Models

Every optical surface has a **BSDF** (Bidirectional Scattering Distribution Function)
that controls how rays interact with it. The BSDF determines the direction and
relative flux of scattered rays.

The NSQ engine ships with four built-in BSDFs:

| BSDF | Scatter type | Parameters |
|---|---|---|
| Default (None) | Specular Fresnel refraction/reflection | — |
| `SpecularBRDF` | Perfect mirror reflection | — |
| `LambertianBSDF` | Cosine-weighted diffuse scatter | `reflectance_value` ∈ [0,1] |
| `HarveyShackBSDF` | Micro-roughness scatter (ABg model) | `b0`, `l0`, `s` |
| `TabulatedBSDF` | User-defined lookup table | angles + BRDF values |

BSDFs are attached to surfaces via `SurfaceConfig` inside a `LensConfig`,
`MirrorConfig`, or any other component config.

In [1]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

from optiland.coordinate_system import CoordinateSystem
from optiland.nonsequential import (
    NSQScene, Spectrum,
    CollimatedSourceConfig, PointSourceConfig,
    IrradianceDetectorConfig,
    LensConfig, MirrorConfig,
    SurfaceConfig,
    SpecularBRDF, LambertianBSDF, HarveyShackBSDF,
)

spec = Spectrum.monochromatic(0.55)

## 1. Default Behaviour — Fresnel Refraction

When no BSDF is set, a refractive surface uses **detached-sample / attached-weight Fresnel splitting**:
each ray is stochastically either refracted or reflected according to the Fresnel
equations, with an attached throughput weight that keeps the estimate unbiased (and differentiable on the torch backend). This is the physically correct treatment for uncoated glass surfaces.

At normal incidence, N-BK7 reflects ~4% per surface (≈8% for both faces of a lens).
Let us verify this with flux bookkeeping:

In [2]:
scene = NSQScene()
scene.add_source(
    'S', CoordinateSystem(z=-80),
    CollimatedSourceConfig(spectrum=spec, total_flux=1.0, aperture_radius=10.0),
)
scene.add_lens(
    'L', CoordinateSystem(z=0),
    LensConfig(r1=50, r2=-50, thickness=5, material='N-BK7', front_aperture_radius=12.5),
)
scene.add_detector(
    'D', CoordinateSystem(z=100),
    IrradianceDetectorConfig(width=20, height=20, num_pixels_x=128, num_pixels_y=128),
)

result = scene.trace(num_rays=50_000, seed=42)
irr = result.detectors['D']

transmission = irr.total_flux / result.total_flux_in * 100
print(f"Flux in          : {result.total_flux_in:.4f} W")
print(f"Flux detected    : {irr.total_flux:.4f} W")
print(f"Transmission     : {transmission:.1f}%")
print(f"Fresnel loss     : {100-transmission:.1f}% (expected ~8% for two uncoated surfaces)")

Flux in          : 1.0000 W
Flux detected    : 0.8656 W
Transmission     : 86.6%
Fresnel loss     : 13.4% (expected ~8% for two uncoated surfaces)


## 2. SpecularBRDF — Perfect Mirror

`SpecularBRDF` forces a surface to reflect all rays perfectly with no transmission
or scatter. This overrides the default Fresnel calculation.

It is useful for ideal mirrors or for temporarily making a surface reflective
without setting `InteractionType.REFLECTIVE` (which also changes how the normal
is computed).

In [3]:
# Mirror using SpecularBRDF
scene_spec = NSQScene()
scene_spec.add_source(
    'S', CoordinateSystem(z=-80),
    CollimatedSourceConfig(spectrum=spec, total_flux=1.0, aperture_radius=15.0),
)
scene_spec.add_mirror(
    'M', CoordinateSystem(z=0, rx=np.pi),
    MirrorConfig(radius=-200, conic=-1.0, aperture_radius=20.0,
                 surface=SurfaceConfig(bsdf=SpecularBRDF())),
)
scene_spec.add_detector(
    'D', CoordinateSystem(z=-100),
    IrradianceDetectorConfig(width=5, height=5, num_pixels_x=128, num_pixels_y=128),
)

result_spec = scene_spec.trace(num_rays=30_000, seed=42)
irr_spec = result_spec.detectors['D']

fig = irr_spec.plot(cmap='hot')
plt.title(f'Parabolic mirror — SpecularBRDF | {irr_spec.num_rays_hit:,} rays')
plt.tight_layout()
plt.show()
plt.close(fig)

C:\Users\kdani\AppData\Local\Temp\ipykernel_23084\3576131454.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. LambertianBSDF — Diffuse Scatter

`LambertianBSDF(reflectance_value)` scatters rays into a cosine-weighted
hemisphere around the surface normal. The `reflectance_value` ∈ [0, 1] scales
the flux of each scattered ray.

A perfect diffuse white surface has `reflectance_value=1.0`. Setting it to 0.8
means 20% of flux is absorbed at each scatter event.

Here we attach a Lambertian BSDF to the *back face* of a lens to scatter some
light backwards — simulating a diffuse rear coating:

In [4]:
lambertian = LambertianBSDF(reflectance_value=0.5)

scene_lamb = NSQScene()
scene_lamb.add_source(
    'S', CoordinateSystem(z=-80),
    CollimatedSourceConfig(spectrum=spec, total_flux=1.0, aperture_radius=10.0),
)
scene_lamb.add_lens(
    'L', CoordinateSystem(z=0),
    LensConfig(
        r1=50, r2=-50, thickness=5, material='N-BK7',
        front_aperture_radius=12.5,
        back=SurfaceConfig(bsdf=lambertian),  # diffuse back face
    ),
)
# Two detectors: one forward and one backward
scene_lamb.add_detector(
    'D_fwd', CoordinateSystem(z=100),
    IrradianceDetectorConfig(width=30, height=30, num_pixels_x=128, num_pixels_y=128),
)
scene_lamb.add_detector(
    'D_bwd', CoordinateSystem(z=-90),
    IrradianceDetectorConfig(width=30, height=30, num_pixels_x=128, num_pixels_y=128),
)

result_lamb = scene_lamb.trace(num_rays=50_000, seed=42)

fwd = result_lamb.detectors['D_fwd']
bwd = result_lamb.detectors['D_bwd']

print(f"Forward detector : {fwd.num_rays_hit:,} rays, {fwd.total_flux:.4f} W")
print(f"Backward detector: {bwd.num_rays_hit:,} rays, {bwd.total_flux:.4f} W")
print(f"Back-scatter fraction: {bwd.total_flux / result_lamb.total_flux_in * 100:.1f}%")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, irr, title in zip(axes, [fwd, bwd], ['Forward (transmitted)', 'Backward (scattered)']):
    im = ax.imshow(irr.irradiance, origin='lower', cmap='hot', aspect='equal',
                   extent=[irr.x_coords[0], irr.x_coords[-1],
                            irr.y_coords[0], irr.y_coords[-1]])
    plt.colorbar(im, ax=ax, label='W/mm²')
    ax.set_title(title)
    ax.set_xlabel('x [mm]'); ax.set_ylabel('y [mm]')
plt.suptitle('LambertianBSDF on lens back face', fontsize=12)
plt.tight_layout()
plt.show()
plt.close(fig)

C:\Users\kdani\Documents\Python_Scripts\optiland\optiland\nonsequential\bsdf\lambertian.py:127: RuntimeWarning: invalid value encountered in divide
  t_vec /= (t_vec * t_vec).sum(axis=1, keepdims=True) ** 0.5


Forward detector : 0 rays, 0.0000 W
Backward detector: 3,226 rays, 0.0341 W
Back-scatter fraction: 3.4%


C:\Users\kdani\AppData\Local\Temp\ipykernel_23084\3494645469.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. HarveyShackBSDF — Micro-Roughness Scatter

The **Harvey-Shack ABg model** is widely used in optical engineering for surface
micro-roughness scatter. The BSDF is:

$$\text{BSDF}(|\beta - \beta_0|) = \frac{b_0}{1 + |\beta - \beta_0|/l_0)^s}$$

where β and β₀ are the scattered and specular direction cosines.

Parameters:
- `b0` — scatter amplitude at zero angle [sr⁻¹]
- `l0` — break-frequency in direction-cosine space (dimensionless)
- `s` — roll-off slope exponent (positive)

A highly polished surface has small `b0` (low scatter level). A ground surface
has large `b0`.

In [5]:
# Compare a polished and a rough mirror surface
polished = HarveyShackBSDF(b0=1e-4, l0=0.01, s=2.0)   # smooth, low scatter
rough    = HarveyShackBSDF(b0=1e-2, l0=0.05, s=1.5)   # rougher, more scatter

def trace_mirror_bsdf(bsdf, n_rays=30_000):
    scene_bsdf = NSQScene()
    scene_bsdf.add_source(
        'S', CoordinateSystem(z=-80),
        CollimatedSourceConfig(spectrum=spec, total_flux=1.0, aperture_radius=15.0),
    )
    scene_bsdf.add_mirror(
        'M', CoordinateSystem(z=0, rx=np.pi),
        MirrorConfig(radius=-200, conic=-1.0, aperture_radius=20.0,
                     surface=SurfaceConfig(bsdf=bsdf)),
    )
    scene_bsdf.add_detector(
        'D', CoordinateSystem(z=-100),
        IrradianceDetectorConfig(width=10, height=10, num_pixels_x=128, num_pixels_y=128),
    )
    res = scene_bsdf.trace(num_rays=n_rays, seed=42)
    return res.detectors['D']

irr_polished = trace_mirror_bsdf(polished)
irr_rough    = trace_mirror_bsdf(rough)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, irr, title in zip(axes, [irr_polished, irr_rough], ['Polished', 'Rough']):
    im = ax.imshow(irr.irradiance, origin='lower', cmap='hot', aspect='equal',
                   extent=[irr.x_coords[0], irr.x_coords[-1],
                            irr.y_coords[0], irr.y_coords[-1]])
    plt.colorbar(im, ax=ax, label='W/mm²')
    ax.set_title(f'HarveyShack ({title}) | {irr.num_rays_hit:,} rays')
    ax.set_xlabel('x [mm]'); ax.set_ylabel('y [mm]')
plt.tight_layout()
plt.show()
plt.close(fig)

C:\Users\kdani\AppData\Local\Temp\ipykernel_23084\216046370.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Comparing Scatter Distributions

Use a `RayDatabaseDetector` to directly inspect the scattered ray directions
from a Lambertian surface:

In [6]:
from optiland.nonsequential import RayDatabaseConfig

# Point source hitting a flat Lambertian surface, capturing the scatter
scene_dir = NSQScene()
scene_dir.add_source(
    'S', CoordinateSystem(z=-30),
    PointSourceConfig(spectrum=spec, total_flux=1.0, half_angle_deg=5),
)
# A near-flat plate with a Lambertian front face scatters the incident beam.
scene_dir.add_lens(
    'L', CoordinateSystem(z=0),
    LensConfig(
        r1=1e9, r2=1e9, thickness=1, material='N-BK7',  # flat plate
        front_aperture_radius=5,
        front=SurfaceConfig(bsdf=LambertianBSDF(1.0)),
    ),
)

scene_dir.add_detector(
    'RDB', CoordinateSystem(z=20),
    RayDatabaseConfig(width=60, height=60),
)

result_dir = scene_dir.trace(num_rays=20_000, seed=1)
db2 = result_dir.detectors['RDB']

# Polar angle of scattered rays
theta = np.degrees(np.arccos(np.clip(db2.N, -1, 1)))

fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(theta, bins=45, density=True, label='Simulated scatter')
theta_theory = np.linspace(0, 90, 200)
ax.plot(theta_theory, np.sin(np.radians(theta_theory)) * np.pi / 180,
        'r--', label='Lambertian cos(theta).sin(theta) (theory)')
ax.set_xlabel('Polar angle theta [deg]')
ax.set_ylabel('Probability density')
ax.set_title('Lambertian scatter angle distribution')
ax.legend()
plt.tight_layout()
plt.show()
plt.close(fig)


C:\Users\kdani\Documents\Python_Scripts\optiland\.venv\Lib\site-packages\numpy\lib\_histograms_impl.py:897: RuntimeWarning: invalid value encountered in divide
  return n / db / n.sum(), bin_edges
C:\Users\kdani\AppData\Local\Temp\ipykernel_23084\1158366716.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Summary

- Default refractive surfaces use probabilistic Fresnel splitting (no BSDF needed)
- `SpecularBRDF()` — perfect mirror; no transmission, no scatter
- `LambertianBSDF(reflectance_value)` — cosine-weighted hemisphere; models diffuse surfaces
- `HarveyShackBSDF(b0, l0, s)` — ABg roughness model; models polished-to-ground surfaces
- Attach a BSDF via `SurfaceConfig(bsdf=...)` inside any component config
- Use `RayDatabaseDetector` to inspect scatter directions directly